In [22]:
# !pip install -U huggingface_hub --break-system-packages

In [ ]:
# !pip install --upgrade transformers --break-system-packages

In [ ]:
# !pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [ ]:
!hf download Qwen/Qwen3.5-0.8B --local-dir ./local_model

Fetching 13 files:   0%|                                 | 0/13 [00:00<?, ?it/s]Still waiting to acquire lock on local_qwen_0.8b/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on local_qwen_0.8b/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on local_qwen_0.8b/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Fetching 13 files: 100%|████████████████████████| 13/13 [00:05<00:00,  2.25it/s]
Download complete: : 10.2MB [00:05, 11.9MB/s]              /workspace/local_qwen_0.8b
Download complete: : 10.2MB [00:05, 1.74MB/s]


In [ ]:
from safetensors import safe_open
import os

model_dir = "./local_model"

for file in os.listdir(model_dir):
    if file.endswith(".safetensors"):
        path = os.path.join(model_dir, file)
        print(f"\nChecking {file}")

        with safe_open(path, framework="pt") as f:
            keys = list(f.keys())

            mtp_keys = [k for k in keys if "mtp" in k.lower()]
            for k in mtp_keys:
                print(k)


Checking model.safetensors-00001-of-00001.safetensors
mtp.fc.weight
mtp.layers.0.input_layernorm.weight
mtp.layers.0.mlp.down_proj.weight
mtp.layers.0.mlp.gate_proj.weight
mtp.layers.0.mlp.up_proj.weight
mtp.layers.0.post_attention_layernorm.weight
mtp.layers.0.self_attn.k_norm.weight
mtp.layers.0.self_attn.k_proj.weight
mtp.layers.0.self_attn.o_proj.weight
mtp.layers.0.self_attn.q_norm.weight
mtp.layers.0.self_attn.q_proj.weight
mtp.layers.0.self_attn.v_proj.weight
mtp.norm.weight
mtp.pre_fc_norm_embedding.weight
mtp.pre_fc_norm_hidden.weight


In [8]:
import os
import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, notebook_login, get_token
from transformers import AutoModelForImageTextToText, AutoProcessor

In [9]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [10]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.10.0+cu126
CUDA Available: True
CUDA Version: 12.6
GPU Name: NVIDIA RTX A5000
VRAM: 23.6 GB


In [11]:
MODEL_ID = "Qwen/Qwen3.5-0.8B"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"

In [12]:
notebook_login()

In [13]:
model = AutoModelForImageTextToText.from_pretrained(
    model_dir, 
    dtype=torch.bfloat16, 
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_dir)

tokenizer = processor.tokenizer


2026-03-15 15:08:14 WARNING modeling_qwen3_5.py L501: The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [14]:
model

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 768)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-11): 12 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=768, out_features=3072, bias=True)
            (linear_fc2): Linear(in_features=3072, out_features=768, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
      (mer

In [ ]:
TUNING_CONFIG = {
    "group_size": 128,
    "sym": True,
    "iters": 10,  # High accuracy (Production grade)
    "nsamples": 32,  # More calibration data
    "batch_size": 2,  # Faster on 48GB VRAM
    "seqlen": 2048,
    "low_gpu_mem_usage": False,  # Keep on GPU for speed
    "enable_torch_compile": True,  # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
    "layer_config": {
        "mtp": {"data_type": "bfloat16"},
        "mtp.fc": {"data_type": "bfloat16"}
    }
}

In [16]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [17]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W4A16",
    **TUNING_CONFIG,
)

2026-03-15 15:09:00 INFO autoround.py L165: using MLLM mode for multimodal model.
2026-03-15 15:09:02 INFO base.py L504: using torch.bfloat16 for quantization tuning


In [18]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round", inplace=True
)

2026-03-15 15:09:03 WARNING formats.py L154: some layers are skipped quantization (shape not divisible by 32).
2026-03-15 15:09:03 WARNING modeling_utils.py L4430: `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
2026-03-15 15:09:03 INFO base.py L1784: start to cache block inputs


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1229 [00:00<?, ? examples/s]

cache block inputs: 100%|██████████| 32/32 [00:00<00:00, 70.21it/s] 
2026-03-15 15:09:54 INFO base.py L1801: caching done
2026-03-15 15:09:54 INFO offload.py L346: offloading module weights...
2026-03-15 15:09:54 INFO offload.py L610: OffloadManager (compressor): tempdir = /tmp/compressor_bjq51cof
2026-03-15 15:09:56 INFO offload.py L360: offload done, freed 0.93 GB
Quantizing model.language_model.layers.0:   0%|          | 0/24 [00:00<?, ?it/s]2026-03-15 15:10:01 INFO base.py L3147: ['linear_attn.in_proj_b', 'linear_attn.in_proj_a'] have not been quantized
W0315 15:10:01.675000 14794 torch/_dynamo/convert_frame.py:1676] [0/8] torch._dynamo hit config.recompile_limit (8)
W0315 15:10:01.675000 14794 torch/_dynamo/convert_frame.py:1676] [0/8]    function: 'quant_tensor_sym' (/usr/local/lib/python3.12/dist-packages/auto_round/data_type/int.py:118)
W0315 15:10:01.675000 14794 torch/_dynamo/convert_frame.py:1676] [0/8]    last reason: 0/7: tensor 'v' size mismatch at index 0. expected 16384

(Qwen3_5ForConditionalGeneration(
   (model): Qwen3_5Model(
     (visual): Qwen3_5VisionModel(
       (patch_embed): Qwen3_5VisionPatchEmbed(
         (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
       )
       (pos_embed): Embedding(2304, 768)
       (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
       (blocks): ModuleList(
         (0-11): 12 x Qwen3_5VisionBlock(
           (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
           (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
           (attn): Qwen3_5VisionAttention(
             (qkv): Linear(in_features=768, out_features=2304, bias=True)
             (proj): Linear(in_features=768, out_features=768, bias=True)
           )
           (mlp): Qwen3_5VisionMLP(
             (linear_fc1): Linear(in_features=768, out_features=3072, bias=True)
             (linear_fc2): Linear(in_features=3072, out_features=768, bias=True)
             (act_fn): GELUTanh()
           )
       

In [19]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [20]:
path_autoround = os.path.join(OUTPUT_BASE_DIR, "auto-round-auto-gptq")
path_gptq = os.path.join(OUTPUT_BASE_DIR, "auto-gptq")
path_awq = os.path.join(OUTPUT_BASE_DIR, "auto-awq")

In [ ]:
if hf_token:
    # 1. AutoRound Repo
    # Verify path exists before uploading
    if os.path.exists(path_autoround):
        push_to_hub(path_autoround, f"{base_name}-W4A16-AutoRound", hf_token)
    else:
        print(f"⚠️ Could not find AutoRound output at {path_autoround}")

    # 2. GPTQ Repo
    if os.path.exists(path_gptq):
        push_to_hub(path_gptq, f"{base_name}-W4A16-AutoRound-GPTQ", hf_token)
    else:
        print(f"⚠️ Could not find GPTQ output at {path_gptq}")

    # 3. AWQ Repo
    if os.path.exists(path_awq):
        push_to_hub(path_awq, f"{base_name}-W4A16-AutoRound-AWQ", hf_token)
    else:
        print(f"⚠️ Could not find AWQ output at {path_awq}")

In [21]:
push_to_hub("./AutoRound", f"{base_name}-W4A16-AutoRound", hf_token)


[Hub] Pushing ./AutoRound to Vishva007/Qwen3.5-0.8B-W4A16-AutoRound...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.5-0.8B-W4A16-AutoRound
